# Pix2Struct DocVQA-base — bounded document-QA adaptation (standalone E2E)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/pix2struct-docvqa-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/pix2struct-docvqa-pipeline/blob/main/tutorials/pix2struct_docvqa_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97-google%2Fpix2struct--docvqa--base-ffcc4d?style=flat)](https://huggingface.co/google/pix2struct-docvqa-base)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** end-to-end adaptation of OCR-free document question answering: validated page/question/answer records → held-out ANLS comparison → reloadable safetensors adapter

**This notebook is standalone.** It carries the repository's package (3 modules under `src/pix2struct_docvqa_pipeline/`, at revision `3fedbaf6f717`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `63f6b3de436e39f75c7a486881a9c2c14a7f4e89` (~1133 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported GPU runtime installs the pinned dependencies, carries the three repository modules without cloning the repository, stages and digest-verifies the immutable model snapshot, builds and validates the deterministic synthetic corpus, records two non-neural baselines and the frozen model, fine-tunes only the final two decoder blocks with validation-ANLS epoch selection, evaluates held-out documents, exports a safetensors adapter bound to the base digest, reloads it in a fresh model instance, checks answer parity, and writes machine-readable evidence. CUDA is required; no credential or upload is needed.

**Bring Your Own Data:** After the default sample completes, set `USE_BYOD = True` and upload one zip containing `records.csv` plus its page images. CSV columns are `id,document_id,file,question,answers`; separate accepted answers with `|`. The same validation and document-grouped split apply. BYOD is optional and remains inside the hosted runtime.

Pix2Struct renders each question above its page, encodes the composite as at most 2,048 patches, and generates a short answer. This tutorial makes that path trainable without changing its inference contract: the vision encoder, embeddings, language head and first ten decoder blocks stay frozen; only decoder blocks 10 and 11 are updated. A deterministic, code-generated business-document corpus avoids registrations, remote dataset drift and private data. Documents—not question rows—define the train/validation/test boundary. Validation ANLS selects the epoch; the test set is used only for frozen/adapted comparison. This carrier remains Candidate until its exact committed blob passes a clean supported GPU run.

**Learning objectives:** verify immutable model and corpus identities; inspect record and split contracts; compare empty-answer and training-majority baselines with the frozen model; run bounded decoder adaptation; interpret ANLS and exact match as one seeded synthetic-domain estimate; and export, verify and reload the adapter without pickle.

**This notebook does not demonstrate:** DocVQA benchmark claims, PDFs or multi-page reasoning, OCR boxes or answer localisation, confidence calibration, full-model fine-tuning, hyperparameter search, production throughput, and local execution evidence. The synthetic corpus tests a narrow invoice-like domain and cannot establish real-scan performance.

## Prerequisites

- **Runtime:** a fresh Google Colab or Kaggle-style Python 3.12 runtime with a CUDA GPU. The notebook refuses CPU for adaptation. The pinned checkpoint is about 1.13 GB and is downloaded at its immutable revision, then checked against the embedded manifest.
- **Knowledge:** Python, supervised train/validation/test splits, autoregressive generation, and why held-out ANLS is not a confidence score.
- **Data:** the default path generates 40 fictional documents and 120 question rows in code under CC0-1.0: 72 train, 18 validation and 30 held-out test rows. Optional BYOD must contain only documents you are authorized to process. Do not upload confidential or restricted data to a hosted notebook.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/pix2struct-docvqa-base` snapshot (~1133 MB in total) at revision `63f6b3de436e…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'pix2struct-docvqa-pipeline',
    'repository_revision': '3fedbaf6f71763ed21538c851c36b09971d1ac97',
    'embedded_module': 'src/pix2struct_docvqa_pipeline/pipeline.py',
    'embedded_modules': ['src/pix2struct_docvqa_pipeline/pipeline.py', 'src/pix2struct_docvqa_pipeline/metrics.py', 'src/pix2struct_docvqa_pipeline/samples.py'],
    'module_sha256': 'ab6112d33963d530ba5603808763ac8bf2ce694c3e6517ec31c713051273dff8',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/pix2struct_docvqa_pipeline/` @ `3fedbaf6f717`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/pix2struct_docvqa_pipeline/pipeline.py`

In [ ]:
"""OCR-free document question answering with the pinned ``google/pix2struct-docvqa-base`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Pix2Struct architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed. The question is rendered as a
text header on top of the page (the Pix2Struct VQA input convention) with Pillow's bundled font, so no
font is fetched from the Hub at inference time.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image, ImageFont

MODEL_ID = "google/pix2struct-docvqa-base"
MODEL_REVISION = "63f6b3de436e39f75c7a486881a9c2c14a7f4e89"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "pix2struct-docvqa-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = "067f7f314d87fa56daa5bcfaf36fa0b33ceebf7b7d4fae6a1e51ab7af64ee0b5"
DECODER_LAYERS = 12
DEFAULT_TRAINABLE_DECODER_LAYERS = 2
MAX_TARGET_TOKENS = 32
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50
ARTIFACT_FORMAT = "org.valcorza.pix2struct-docvqa-base.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"

# Generation ceilings. DocVQA answers are short spans (the checkpoint's text_config max_length is 20);
# the default leaves room for a long address or title, the ceiling bounds runaway generation.
MAX_NEW_TOKENS = 128
DEFAULT_MAX_NEW_TOKENS = 32
DECODING = "greedy"
# Question ceiling. The question is rendered as a header line (wrapped at 80 characters by the
# processor) above the page; a very long question shrinks the page's share of the patch budget.
MAX_QUESTION_CHARS = 256
# Input ceilings. The processor extracts at most MAX_PATCHES 16x16 patches (preprocessor_config.json)
# after scaling the image to fill that budget, so pixel count only guards memory during resizing.
MAX_PATCHES = 2048
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
# ANLS (DocVQA's official metric): a normalised Levenshtein similarity below this threshold scores 0.
ANLS_THRESHOLD = 0.5
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def header_font_bytes() -> bytes:
    """Pillow's bundled Aileron Regular (CC0) as TrueType bytes: the header font for the rendered question.

    The upstream image processor otherwise fetches ``ybelkada/fonts/Arial.TTF`` from the Hub at
    inference time — an unpinned, unlisted download of a proprietary font. The bundled subset covers
    the printable ASCII range, which is what a question is expected to use.
    """
    font = ImageFont.load_default(size=36)
    data = getattr(font, "font_bytes", None)
    if not data:
        raise RuntimeError("Pillow's bundled TrueType font is unavailable (FreeType support missing)")
    return bytes(data)


def normalize_answer(text: str) -> str:
    """DocVQA-style normalisation: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def _levenshtein(a: str, b: str) -> int:
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def anls(prediction: str, golds: Sequence[str], *, threshold: float = ANLS_THRESHOLD) -> float:
    """Average Normalised Levenshtein Similarity for one question (Biten et al., ICDAR 2019).

    ``1 - lev(pred, gold) / max(len(pred), len(gold))`` over normalised strings, maximised over the
    accepted ``golds``; a similarity below ``threshold`` scores 0 so a near-miss is not rewarded.
    """
    if not golds:
        raise ValueError("golds must contain at least one accepted answer")
    pred = normalize_answer(prediction)
    best = 0.0
    for gold in golds:
        ref = normalize_answer(gold)
        longest = max(len(pred), len(ref))
        similarity = 1.0 if longest == 0 else 1.0 - _levenshtein(pred, ref) / longest
        best = max(best, similarity)
    return best if best >= threshold else 0.0


def exact_match(prediction: str, golds: Sequence[str]) -> bool:
    """Whether the normalised prediction equals any normalised accepted answer."""
    pred = normalize_answer(prediction)
    return any(pred == normalize_answer(gold) for gold in golds)


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one page image as PIL.Image.Image (any mode, converted to RGB) plus one question string",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "question_chars": [1, MAX_QUESTION_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False), deterministic on a fixed device and dtype",
    "preprocessing": (
        "the question is rendered as a black-on-white header (Pillow's bundled font, wrapped at 80 "
        "characters) above the page; the composite is scaled to fill at most MAX_PATCHES 16x16 patches "
        "(aspect ratio preserved), normalised per image, and flattened into patch tokens with row/column "
        "positions; the decoder generates the answer text"
    ),
    "output": "one answer string (the model's decoded text), no score",
}


def _check_inputs(image: Any, question: Any, max_new_tokens: Any) -> tuple[Image.Image, str, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``answer`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    if not isinstance(question, str):
        raise TypeError("question must be a str")
    checked_question = " ".join(question.split())
    if not checked_question:
        raise ValueError("question must contain at least one non-whitespace character")
    if len(checked_question) > MAX_QUESTION_CHARS:
        raise ValueError(
            f"question has {len(checked_question)} chars > MAX_QUESTION_CHARS {MAX_QUESTION_CHARS}"
        )
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, checked_question, max_new_tokens


def validate_inputs(
    image: Image.Image,
    questions: Sequence[str],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every question is checked exactly as ``answer`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(questions, str) or not isinstance(questions, Sequence) or not questions:
        raise TypeError("questions must be a non-empty sequence of str")
    checked = [_check_inputs(image, question, max_new_tokens)[1] for question in questions]
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (answer takes one page image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "questions": checked,
        "generation": {"max_new_tokens": int(max_new_tokens), "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    golds: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``golds`` (one sequence of accepted answers per result, in order) the report carries the
    mean ``anls`` and the ``exact_match`` rate over the questions plus one per-question entry, verdict
    ``sample-sanity``; without golds it is ``not-measurable`` and says what labelled data would make
    the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one answer result")
    base = {
        "task": "document page image + question -> answer text (OCR-free)",
        "score_semantics": (
            "the answer is generated text and carries no score, probability or correctness signal; a "
            "fluent answer is not evidence that it is read from the page. Greedy decoding makes the "
            "output reproducible on a fixed device and dtype, a reproducibility property, not a quality one"
        ),
        "sample_kind": sample_kind,
        "n_questions": len(results),
        "truncated": [bool(result.get("truncated")) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if golds is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no accepted answers were supplied for the evaluated questions",
            "needs": (
                "question/answer pairs with accepted answers on pages from the deployment domain "
                "(DocVQA-style annotations) scored with ANLS; no such labelled set ships with this repository"
            ),
        }
    if len(golds) != len(results):
        raise ValueError(f"golds has {len(golds)} entries for {len(results)} results")
    per_question = []
    for result, accepted in zip(results, golds, strict=True):
        if isinstance(accepted, str) or not accepted:
            raise ValueError("each golds entry must be a non-empty sequence of accepted answers")
        prediction = str(result["answer"])
        per_question.append(
            {
                "question": result.get("question"),
                "prediction": prediction,
                "golds": list(accepted),
                "anls": anls(prediction, accepted),
                "exact_match": exact_match(prediction, accepted),
            }
        )
    metrics = [
        {
            "id": "anls",
            "value": sum(entry["anls"] for entry in per_question) / len(per_question),
            "threshold": ANLS_THRESHOLD,
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; max over golds",
            "estimation": f"{len(per_question)} question(s) on one page, no dispersion estimate",
        },
        {
            "id": "exact_match",
            "value": sum(entry["exact_match"] for entry in per_question) / len(per_question),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed",
            "estimation": f"{len(per_question)} question(s) on one page, no dispersion estimate",
        },
    ]
    return {
        **base,
        "metrics": metrics,
        "per_question": per_question,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_question)} authored question(s) on one tutorial page whose content you rendered "
            "yourself; plumbing evidence, not a DocVQA benchmark"
        ),
        "needs": (
            "a labelled question/answer set on pages from the deployment domain (scans, forms, layouts) "
            "for any accuracy claim; the DocVQA benchmark itself is registration-gated and not bundled"
        ),
    }


@dataclass
class Pix2StructDocVQAPipeline:
    """``_runner(image, question, max_new_tokens)`` returns ``{"answer": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    _font_bytes: bytes | None = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructDocVQAPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        font_bytes = header_font_bytes()
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Pix2StructProcessor.from_pretrained(location, **common)
        if not getattr(processor.image_processor, "is_vqa", False):
            raise RuntimeError("snapshot image processor is not the VQA variant (is_vqa=False); refusing")
        model = Pix2StructForConditionalGeneration.from_pretrained(location, dtype=torch.float32, **common)
        model = model.eval().to(resolved_device)
        for param in model.parameters():
            param.requires_grad_(False)

        def runner(image: Image.Image, question: str, max_new_tokens: int) -> dict[str, Any]:
            # The image processor is called directly: Pix2StructProcessor.__call__ drops the
            # font_bytes kwarg, and font_bytes is what replaces the default Hub font download
            # (see header_font_bytes). The VQA processor renders the question as the header.
            inputs = processor.image_processor(
                image, header_text=question, return_tensors="pt", font_bytes=font_bytes
            ).to(resolved_device)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            # Encoder-decoder: the output holds only decoder tokens (decoder_start + answer + eos).
            answer_ids = generated[0]
            decoded = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
            return {"answer": decoded, "new_tokens": int(answer_ids.shape[0]) - 1}

        return cls(
            runner,
            resolved_device,
            "float32",
            source,
            _model=model,
            _processor=processor,
            _font_bytes=font_bytes,
        )

    def answer(
        self,
        image: Image.Image,
        question: str,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Answer one question about one page image; ``answer`` is the decoded text, stripped."""
        rgb, checked_question, checked_tokens = _check_inputs(image, question, max_new_tokens)
        raw = self._runner(rgb, checked_question, checked_tokens)
        if not isinstance(raw, dict) or "answer" not in raw:
            raise RuntimeError("runner must return a dict with 'answer'")
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "answer": str(raw["answer"]).strip(),
            "question": checked_question,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation contract -----------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None or self._font_bytes is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._processor

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Answer and score a validated document-QA corpus with ANLS and exact match."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import qa_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions = [
            self.answer(record["image"], record["question"], max_new_tokens=max_new_tokens)["answer"]
            for record in checked
        ]
        metrics = qa_metrics(predictions, checked)
        metrics.update(
            {
                "max_new_tokens": max_new_tokens,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _trainable_names(self, trainable_decoder_layers: int) -> list[str]:
        if (
            isinstance(trainable_decoder_layers, bool)
            or not isinstance(trainable_decoder_layers, int)
            or not 1 <= trainable_decoder_layers <= DECODER_LAYERS
        ):
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        first = DECODER_LAYERS - trainable_decoder_layers
        prefixes = tuple(f"decoder.layer.{index}." for index in range(first, DECODER_LAYERS))
        names = [name for name, _param in model.named_parameters() if name.startswith(prefixes)]
        if not names:
            raise RuntimeError("no decoder-layer parameters matched the pinned Pix2Struct architecture")
        return names

    def _training_inputs(self, records: Sequence[Mapping[str, Any]]) -> Any:
        model, processor = self._require_model()
        device = next(model.parameters()).device
        return processor.image_processor(
            images=[record["image"] for record in records],
            header_text=[record["question"] for record in records],
            max_patches=MAX_PATCHES,
            return_tensors="pt",
            font_bytes=self._font_bytes,
        ).to(device)

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 3,
        lr: float = 2e-4,
        batch_size: int = 1,
        trainable_decoder_layers: int = DEFAULT_TRAINABLE_DECODER_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Tune only the last decoder blocks and retain the best validation-ANLS epoch."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not 0.0 < lr <= 1e-3:
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 16:
            raise ValueError("batch_size must be an int in 1..16")
        names = self._trainable_names(trainable_decoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )

        import torch

        torch.manual_seed(seed)
        model, processor = self._require_model()
        device = next(model.parameters()).device
        wanted = set(names)
        initial_state = {
            name: value.detach().clone()
            for name, value in model.state_dict().items()
            if name in wanted
        }
        best_state = {name: value.clone() for name, value in initial_state.items()}
        started = time.perf_counter()

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                key: value
                for key, value in self.evaluate(val_checked).items()
                if key in ("anls", "exact_match", "n")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {
            "epoch": 0,
            "train_loss": None,
            "val": score_val(),
            "note": "frozen model",
        }
        history.append(entry)
        if progress:
            progress(entry)
        best_score = entry["val"]["anls"] if entry["val"] else -math.inf
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [param for param in model.parameters() if param.requires_grad]
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                model.encoder.eval()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    indexes = order[start : start + batch_size]
                    batch = [train_checked[index] for index in indexes]
                    inputs = self._training_inputs(batch)
                    tokenized = processor.tokenizer(
                        [record["answers"][0] for record in batch],
                        padding=True,
                        truncation=True,
                        max_length=MAX_TARGET_TOKENS,
                        return_tensors="pt",
                    ).to(device)
                    labels = tokenized["input_ids"].clone()
                    labels[labels == processor.tokenizer.pad_token_id] = -100
                    optimiser.zero_grad(set_to_none=True)
                    output = model(**inputs, labels=labels)
                    output.loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(output.loss.detach()))
                model.eval()
                entry = {
                    "epoch": epoch,
                    "train_loss": sum(losses) / len(losses),
                    "val": score_val(),
                }
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["anls"] if entry["val"] else math.inf
                if current > best_score or not entry["val"]:
                    best_score = current
                    best_state = {
                        name: value.detach().clone()
                        for name, value in model.state_dict().items()
                        if name in wanted
                    }
                    best_epoch = epoch
        except BaseException:
            restored = dict(model.state_dict())
            restored.update(initial_state)
            model.load_state_dict(restored, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_decoder_layers": trainable_decoder_layers,
            "trainable_names": names,
            "n_trainable": sum(param.numel() for param in params),
            "n_total": sum(param.numel() for param in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation ANLS" if val_checked else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ---------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Save the adapted decoder tensors as safetensors, bound to the pinned base digest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {
            name: value.detach().cpu().contiguous()
            for name, value in model.state_dict().items()
            if name in names
        }
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {
                key: value
                for key, value in self.adapter.items()
                if key not in ("history", "trainable_names")
            },
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file") != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_decoder_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int) or not 1 <= layers <= DECODER_LAYERS:
            raise ValueError("artifact manifest does not record valid trainable decoder layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify manifest, digest and exact tensor set before loading an adapter."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_decoder_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("decoder.layer."):
                raise ValueError(f"artifact tensor {key} is not an adaptable decoder tensor")
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({key: value.to(state[key].dtype) for key, value in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructDocVQAPipeline:
        pipeline = cls.from_pretrained(
            device=device,
            weights_dir=weights_dir,
            allow_download=allow_download,
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/pix2struct_docvqa_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus metrics and non-adapted baselines for document visual question answering."""

from __future__ import annotations

import statistics
from collections import Counter
from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import anls, exact_match` removed — names are kernel globals defined by the carried modules

METRIC_DEFINITIONS = {
    "anls": "mean answer normalized Levenshtein similarity with similarities below 0.5 set to zero",
    "exact_match": "fraction whose normalized prediction equals an accepted answer",
}


def qa_metrics(predictions: Sequence[str], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    if len(predictions) != len(records):
        raise ValueError(f"{len(predictions)} predictions for {len(records)} records")
    if not records:
        raise ValueError("records must not be empty")
    rows = []
    for prediction, record in zip(predictions, records, strict=True):
        answers = list(record["answers"])
        rows.append(
            {
                "id": record["id"],
                "document_id": record["document_id"],
                "question": record["question"],
                "prediction": str(prediction),
                "answers": answers,
                "anls": anls(str(prediction), answers),
                "exact_match": exact_match(str(prediction), answers),
            }
        )
    return {
        "n": len(rows),
        "anls": round(statistics.fmean(row["anls"] for row in rows), 4),
        "exact_match": round(statistics.fmean(float(row["exact_match"]) for row in rows), 4),
        "rows": rows,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def empty_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    out = qa_metrics([""] * len(records), records)
    out["baseline"] = "empty answer"
    return out


def majority_answer_baseline(
    records: Sequence[Mapping[str, Any]], reference: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    if not reference:
        raise ValueError("reference records must not be empty")
    answer = Counter(str(r["answers"][0]) for r in reference).most_common(1)[0][0]
    out = qa_metrics([answer] * len(records), records)
    out["baseline"] = "most frequent normalized training answer, ignoring image and question"
    out["answer"] = answer
    return out

**Module 3/3:** `src/pix2struct_docvqa_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic synthetic document-QA data and BYOD validation for the E2E carrier."""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image, ImageDraw, ImageFont

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_QUESTION_CHARS, normalize_answer, validate_image` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "DIMER synthetic business-document QA sample"
CORPUS_LICENSE = "CC0-1.0 (generated in code; no external document or personal data)"
CORPUS_VERSION = "1"
SAMPLE_SEED = 42
SAMPLE_DOCUMENTS = {"train": 24, "validation": 6, "test": 10}
QUESTIONS_PER_DOCUMENT = 3
SAMPLE_SPLIT = {name: count * QUESTIONS_PER_DOCUMENT for name, count in SAMPLE_DOCUMENTS.items()}
SAMPLE_DIGEST = "b9e7b0b27ae120979c988da745c8d0324de4267925f8c53b7a2f7f66317a438a"
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MAX_ANSWERS = 8
MAX_ANSWER_CHARS = 128
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")

_VENDORS = ("Northwind Office", "Blue Fern Bakery", "Harbor Tools", "Cedar Health", "Atlas Transit")
_CITIES = ("Manila", "Cebu", "Davao", "Baguio", "Iloilo", "Quezon City")
_CONTACTS = ("Ana Reyes", "Miguel Santos", "Lea Cruz", "Paolo Lim", "Maya Flores")


def _sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def image_digest(image: Image.Image) -> str:
    rgb = image.convert("RGB")
    return _sha256(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    parts = sorted(
        f"{r['id']}:{r['document_id']}:{image_digest(r['image'])}:{r['question']}:{'|'.join(r['answers'])}"
        for r in records
    )
    return _sha256("\n".join(parts).encode())


def _font(size: int) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    return ImageFont.load_default(size=size)


def _document(index: int) -> tuple[Image.Image, dict[str, str]]:
    vendor = _VENDORS[index % len(_VENDORS)]
    city = _CITIES[(index * 3) % len(_CITIES)]
    contact = _CONTACTS[(index * 2) % len(_CONTACTS)]
    invoice = f"INV-{2026 + index // 90}-{1000 + index:04d}"
    po = f"PO-{70000 + index * 17}"
    total = f"PHP {1250 + index * 137:,.2f}"
    date = f"2026-{1 + index % 9:02d}-{1 + (index * 7) % 27:02d}"
    fields = {
        "vendor": vendor,
        "city": city,
        "contact": contact,
        "invoice": invoice,
        "po": po,
        "total": total,
        "date": date,
    }
    image = Image.new("RGB", (768, 1024), "white")
    draw = ImageDraw.Draw(image)
    title, body, small = _font(34), _font(24), _font(18)
    accent = (35 + index * 13 % 120, 70 + index * 17 % 120, 120 + index * 19 % 100)
    draw.rectangle((0, 0, 768, 120), fill=accent)
    draw.text((36, 32), vendor.upper(), font=title, fill="white")
    draw.text((36, 82), f"{city} branch", font=small, fill="white")
    if index % 2:
        labels = [
            ("INVOICE", invoice),
            ("DATE", date),
            ("PURCHASE ORDER", po),
            ("CONTACT", contact),
            ("TOTAL DUE", total),
        ]
        y = 180
        for label, value in labels:
            draw.text((48, y), label, font=small, fill="gray")
            draw.text((280, y), value, font=body, fill="black")
            draw.line((48, y + 38, 720, y + 38), fill=(210, 210, 210), width=2)
            y += 105
    else:
        draw.text((48, 175), "INVOICE", font=title, fill=accent)
        draw.text((510, 184), invoice, font=body, fill="black")
        draw.text((48, 270), f"Date: {date}", font=body, fill="black")
        draw.text((48, 330), f"Purchase order: {po}", font=body, fill="black")
        draw.text((48, 390), f"Contact: {contact}", font=body, fill="black")
        draw.rectangle((430, 540, 720, 640), outline=accent, width=4)
        draw.text((455, 565), f"TOTAL  {total}", font=body, fill="black")
    draw.text((48, 920), "Synthetic training page — no real person or transaction", font=small, fill="gray")
    return image, fields


def build_sample_dataset(seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    order = list(range(sum(SAMPLE_DOCUMENTS.values())))
    random.Random(seed).shuffle(order)
    result: dict[str, list[dict[str, Any]]] = {}
    start = 0
    specs = (
        ("What is the invoice number?", "invoice"),
        ("What is the purchase order number?", "po"),
        ("What is the total due?", "total"),
    )
    for split, n_docs in SAMPLE_DOCUMENTS.items():
        rows = []
        for index in order[start : start + n_docs]:
            image, fields = _document(index)
            document_id = f"synthetic-{index:03d}"
            for q_index, (question, field) in enumerate(specs):
                rows.append(
                    {
                        "id": f"{document_id}-q{q_index}",
                        "document_id": document_id,
                        "image": image.copy(),
                        "question": question,
                        "answers": [fields[field]],
                        "source": "generated",
                    }
                )
        result[split] = rows
        start += n_docs
    return result


def _open(image: Any, where: str) -> Image.Image:
    if isinstance(image, Image.Image):
        return image
    if isinstance(image, str | Path):
        try:
            loaded = Image.open(image)
            loaded.load()
            return loaded
        except Exception as exc:
            raise ValueError(f"{where}: cannot decode image") from exc
    raise ValueError(f"{where}: image must be a PIL image or a path")


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
) -> dict[str, Any]:
    if isinstance(records, Mapping | str | bytes) or not isinstance(records, Sequence):
        raise ValueError("records must be a sequence of document-QA mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} required")
    checked, ids = [], set()
    for index, record in enumerate(records):
        required = ("id", "document_id", "image", "question", "answers")
        if not isinstance(record, Mapping) or any(k not in record for k in required):
            raise ValueError(f"records[{index}] must contain id/document_id/image/question/answers")
        rid, document_id = record["id"], record["document_id"]
        if not isinstance(rid, str) or not _ID_RE.fullmatch(rid) or rid in ids:
            raise ValueError(f"records[{index}]: id must be unique and match {_ID_RE.pattern}")
        if not isinstance(document_id, str) or not _ID_RE.fullmatch(document_id):
            raise ValueError(f"records[{index}]: document_id must match {_ID_RE.pattern}")
        image = validate_image(_open(record["image"], f"records[{index}].image"))
        question = " ".join(str(record["question"]).split())
        if not question or len(question) > MAX_QUESTION_CHARS:
            raise ValueError(f"records[{index}]: question must contain 1..{MAX_QUESTION_CHARS} characters")
        answers = record["answers"]
        invalid_answers = isinstance(answers, str | bytes) or not isinstance(answers, Sequence)
        if invalid_answers or not 1 <= len(answers) <= MAX_ANSWERS:
            raise ValueError(f"records[{index}]: answers must contain 1..{MAX_ANSWERS} strings")
        normalized = []
        for answer in answers:
            if not isinstance(answer, str) or not answer.strip() or len(answer) > MAX_ANSWER_CHARS:
                raise ValueError(
                    f"records[{index}]: each answer must contain 1..{MAX_ANSWER_CHARS} characters"
                )
            if normalize_answer(answer) not in {normalize_answer(a) for a in normalized}:
                normalized.append(answer.strip())
        ids.add(rid)
        checked.append(
            {
                "id": rid,
                "document_id": document_id,
                "image": image,
                "question": question,
                "answers": normalized,
                "source": record.get("source", "byod"),
            }
        )
    return {
        "records": checked,
        "n_records": len(checked),
        "n_documents": len({r["document_id"] for r in checked}),
        "digest": dataset_digest(checked),
    }


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, int]:
    seen: dict[str, str] = {}
    for split, records in splits.items():
        for record in records:
            key = str(record["document_id"])
            if key in seen and seen[key] != split:
                raise ValueError(f"document {key!r} appears in both {seen[key]} and {split}")
            seen[key] = split
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    checked = validate_dataset(records)["records"]
    groups: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        groups.setdefault(record["document_id"], []).append(record)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    n_test, n_val = max(1, round(len(keys) * test_fraction)), round(len(keys) * val_fraction)
    if len(keys) - n_test - n_val < 1:
        raise ValueError("too few distinct documents to split")
    selected = {
        "test": keys[:n_test],
        "validation": keys[n_test : n_test + n_val],
        "train": keys[n_test + n_val :],
    }
    return {name: [row for key in chosen for row in groups[key]] for name, chosen in selected.items()}


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    source = Path(path)
    members: dict[str, bytes] = {}
    if source.is_dir():
        for file in sorted(source.rglob("*")):
            if file.is_file():
                if file.name in members:
                    raise ValueError(f"duplicate basename {file.name!r}")
                members[file.name] = file.read_bytes()
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as archive:
            for info in archive.infolist():
                if not info.is_dir():
                    name = Path(info.filename).name
                    if name in members:
                        raise ValueError(f"duplicate basename {name!r}")
                    members[name] = archive.read(info)
    else:
        raise ValueError(f"{source} is neither a directory nor a zip")
    if "records.csv" not in members:
        raise ValueError("BYOD data must include records.csv")
    rows = list(csv.DictReader(io.StringIO(members["records.csv"].decode("utf-8-sig"))))
    required = ("id", "document_id", "file", "question", "answers")
    if not rows or any(column not in rows[0] for column in required):
        raise ValueError("records.csv must have id, document_id, file, question, answers")
    images: dict[str, Image.Image] = {}
    records = []
    for row in rows:
        name = Path(row["file"]).name
        if name not in members:
            raise ValueError(f"records.csv names missing file {name!r}")
        if name not in images:
            image = Image.open(io.BytesIO(members[name]))
            image.load()
            images[name] = image.convert("RGB")
        records.append(
            {
                "id": row["id"],
                "document_id": row["document_id"],
                "image": images[name].copy(),
                "question": row["question"],
                "answers": [a.strip() for a in row["answers"].split("|") if a.strip()],
                "source": "byod",
            }
        )
    return records


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "document_id", "file", "question", "answers", "source"])
        for record in records:
            writer.writerow(
                [
                    record["id"],
                    record["document_id"],
                    f"{record['document_id']}.png",
                    record["question"],
                    "|".join(record["answers"]),
                    record.get("source", ""),
                ]
            )
    return out


def manifest_json(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> str:
    return json.dumps(
        {
            "corpus": CORPUS_NAME,
            "version": CORPUS_VERSION,
            "license": CORPUS_LICENSE,
            "splits": check_split_disjoint(splits),
            "digest": dataset_digest([record for rows in splits.values() for record in rows]),
        },
        indent=2,
    )

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `63f6b3de436e…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Pix2StructDocVQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "pix2struct-docvqa-base",
  "modelId": "google/pix2struct-docvqa-base",
  "revision": "63f6b3de436e39f75c7a486881a9c2c14a7f4e89",
  "files": [
    {
      "path": "README.md",
      "bytes": 4476,
      "sha256": "794175546e80948e4efef30e95ebe859fdd26d07bcda26f738a1c97feb1e912a"
    },
    {
      "path": "config.json",
      "bytes": 4892,
      "sha256": "8d39973772a4218b555e30daecabdd5ea11aa1345dd711ff7f88fa90750b464f"
    },
    {
      "path": "model.safetensors",
      "bytes": 1129177976,
      "sha256": "067f7f314d87fa56daa5bcfaf36fa0b33ceebf7b7d4fae6a1e51ab7af64ee0b5"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 249,
      "sha256": "c84e4eebc84171d6069533d9f0147ec7b4afd02ab78697cb5c30f9419ef7dc45"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2583,
      "sha256": "5fdb6767a49aca48fdfa43d0279321918185fc4997bdb3ea72bf3a6301a1b43d"
    }
  ],
  "totalBytes": 1133308924
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Pix2StructDocVQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Build or upload the corpus, then validate it

The default corpus has three questions per document and fixed document-level splits. `validate_dataset` checks every id, decoded image, question and accepted answer; `check_split_disjoint` refuses leakage. The aggregate digest must equal `SAMPLE_DIGEST`. Two refusal probes demonstrate duplicate-id and cross-split rejection.

In [ ]:
import json
import os
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42
os.makedirs('outputs', exist_ok=True)

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    upload_name = next(iter(uploaded))
    Path(upload_name).write_bytes(uploaded[upload_name])
    records = load_byod_dataset(upload_name)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD'
else:
    splits = build_sample_dataset()
    data_source = CORPUS_NAME

validated = {name: validate_dataset(rows) for name, rows in splits.items()}
splits = {name: value['records'] for name, value in validated.items()}
split_counts = check_split_disjoint(splits)
all_records = [record for rows in splits.values() for record in rows]
corpus_digest = dataset_digest(all_records)
if not USE_BYOD:
    assert corpus_digest == SAMPLE_DIGEST, (corpus_digest, SAMPLE_DIGEST)
findings = []
try:
    validate_dataset([*splits['train'][:8], {**splits['train'][0]}])
except ValueError as exc:
    findings.append({'probe': 'duplicate-id', 'verdict': 'rejected', 'message': str(exc)})
try:
    check_split_disjoint({'train': splits['train'], 'test': [splits['train'][0]]})
except ValueError as exc:
    findings.append({'probe': 'document-leakage', 'verdict': 'rejected', 'message': str(exc)})
assert len(findings) == 2
input_manifest = {'source': data_source, 'license': CORPUS_LICENSE if not USE_BYOD else 'caller-owned', 'splits': split_counts, 'digest': corpus_digest, 'expected_digest': None if USE_BYOD else SAMPLE_DIGEST, 'document_disjoint': True, 'findings': findings}
Path('outputs/pix2struct_docvqa_input_manifest.json').write_text(json.dumps(input_manifest, indent=2), encoding='utf-8')
print(json.dumps(input_manifest, indent=2))

## 5. Record non-neural baselines and the frozen model

The empty baseline proves the scorer's zero floor. The majority baseline ignores page and question and uses one training-only answer. The frozen checkpoint is evaluated before any update. ANLS gives partial credit only at normalized similarity 0.5 or above; exact match is stricter. These are one-split point estimates.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('The E2E default path requires a CUDA GPU; use a supported GPU runtime.')
gpu_name = torch.cuda.get_device_name(0)
train_records = splits['train']
val_records = splits['validation']
test_records = splits['test']
empty = empty_baseline(test_records)
majority = majority_answer_baseline(test_records, train_records)
frozen = pipe.evaluate(test_records)
print({'gpu': gpu_name, 'empty': {k: empty[k] for k in ('anls', 'exact_match')}, 'majority': {k: majority[k] for k in ('anls', 'exact_match', 'answer')}, 'frozen': {k: frozen[k] for k in ('anls', 'exact_match', 'n', 'seconds')}})

## 6. Adapt the final two decoder blocks

`adapt` freezes every parameter except decoder blocks 10 and 11. AdamW runs for two bounded epochs with gradient clipping; epoch 0 records the frozen validation score and the highest validation ANLS wins. Training is transactional, and the test split is not consulted during selection.

In [ ]:
EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
TRAINABLE_DECODER_LAYERS = 2

def report_epoch(entry):
    print(json.dumps(entry, ensure_ascii=False))

adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_decoder_layers=TRAINABLE_DECODER_LAYERS, seed=0, progress=report_epoch)
assert adapt_result['n_train'] == len(train_records)
assert adapt_result['n_val'] == len(val_records)
assert all(name.startswith(('decoder.layer.10.', 'decoder.layer.11.')) for name in adapt_result['trainable_names'])
print({k: adapt_result[k] for k in ('n_trainable', 'n_total', 'best_epoch', 'selection', 'seconds')})

## 7. Evaluate the selected adapter on held-out documents

The selected model is evaluated once on the held-out test rows. The comparison reports absolute ANLS and exact match plus deltas versus the frozen checkpoint and baselines. Improvement is not an execution gate: regressions remain evidence and must be recorded.

In [ ]:
adapted = pipe.evaluate(test_records)
comparison = {
    'anls': {'empty': empty['anls'], 'majority': majority['anls'], 'frozen': frozen['anls'], 'adapted': adapted['anls']},
    'exact_match': {'empty': empty['exact_match'], 'majority': majority['exact_match'], 'frozen': frozen['exact_match'], 'adapted': adapted['exact_match']},
    'delta_vs_frozen': {'anls': round(adapted['anls'] - frozen['anls'], 4), 'exact_match': round(adapted['exact_match'] - frozen['exact_match'], 4)},
    'n_test': len(test_records),
    'estimation': 'one seeded synthetic document split; no dispersion estimate',
}
print(json.dumps(comparison, indent=2))

## 8. Export, reload, and verify answer parity

The artifact contains only trained decoder tensors in safetensors. Its manifest binds them to the exact base model id, revision and weight digest, plus the artifact size and digest. The live model is released before a fresh base loads the adapter; eight held-out answers must match before results are written.

In [ ]:
import csv
import gc
import platform
import transformers

artifact_dir = Path('outputs/pix2struct_docvqa_adapter')
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'pix2struct_docvqa', 'data_source': data_source, 'corpus_digest': corpus_digest})
parity_records = test_records[:8]
expected_answers = [pipe.answer(record['image'], record['question'])['answer'] for record in parity_records]
del pipe
gc.collect()
torch.cuda.empty_cache()
reloaded = Pix2StructDocVQAPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device='cuda:0')
actual_answers = [reloaded.answer(record['image'], record['question'])['answer'] for record in parity_records]
assert actual_answers == expected_answers
reload_parity = {'identical_answers': len(actual_answers), 'of': len(expected_answers)}
result = {'comparison': comparison, 'frozen': frozen, 'adapted': adapted, 'adaptation': adapt_result, 'reload_parity': reload_parity, 'input_manifest': input_manifest, 'notebook_source': NOTEBOOK_SOURCE, 'repository_revision': NOTEBOOK_SOURCE['repository_revision'], 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'model_license': MODEL_LICENSE, 'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': reloaded.device, 'gpu': gpu_name}}
Path('outputs/pix2struct_docvqa_result.json').write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
with open('outputs/pix2struct_docvqa_predictions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['id', 'document_id', 'question', 'prediction', 'accepted_answers', 'anls', 'exact_match'])
    for row in adapted['rows']:
        writer.writerow([row['id'], row['document_id'], row['question'], row['prediction'], '|'.join(row['answers']), row['anls'], row['exact_match']])
print({'artifact_files': sorted(path.name for path in artifact_dir.iterdir()), 'reload_parity': reload_parity, 'outputs': sorted(os.listdir('outputs'))})

## Interpretation and limits

A successful run proves that this exact carrier can verify the pinned snapshot, create and validate its fixed corpus, measure frozen and adapted held-out behavior, serialize only the intended tensors, and reproduce selected answers after a fresh reload. It does not prove DocVQA benchmark quality, real-document generalization, calibration, robustness to handwriting or unseen layouts, or production fitness. ANLS and exact match are corpus averages, not answer confidence. A negative delta is valid evidence. The generated names and transactions are fictional; BYOD users own consent, licensing, retention and access control.

Successful execution proves that the recorded repository revision, carried in this notebook without the repository being reachable, completes the stated E2E path. It does **not** establish benchmark superiority or production readiness.

**Next experiments:** repeat across seeds; add an external document-domain test set; compare one versus two trainable decoder blocks; stratify by field type and layout; and inspect every held-out error.

## References

- Repository: https://github.com/kurtvalcorza/pix2struct-docvqa-pipeline
- Repository model card: https://github.com/kurtvalcorza/pix2struct-docvqa-pipeline/blob/main/MODEL_CARD.md
- Upstream checkpoint: https://huggingface.co/google/pix2struct-docvqa-base
- Pix2Struct paper: https://arxiv.org/abs/2210.03347
- DocVQA paper: https://arxiv.org/abs/2007.00398
- ANLS origin: https://arxiv.org/abs/1905.13648